# Kafka -> Smart Filtering -> Delta (Databricks)

Notebook nay doc stream tu Kafka, ap dung bo loc luu tru thong minh, va ghi vao Delta table.

## Smart filtering rules
- Threshold: chi luu khi gia tri moi thay doi dang ke so voi gia tri da luu gan nhat.
- Max Interval: neu qua lau du lieu khong doi, van luu 1 ban ghi de giu trace.

Mapping hien tai: 5 sensor, moi sensor do 1 metric:
- sensor_1 -> temperature
- sensor_2 -> humidity
- sensor_3 -> soil_moisture
- sensor_4 -> light_intensity
- sensor_5 -> pressure

In [ ]:
# ===== Runtime configuration =====
# IMPORTANT:
# - Databricks remote cluster will usually NOT reach localhost on your laptop.
# - If using LocaltoNet, keep this endpoint the same as Kafka advertised listener.
USE_LOCALTONET = True
LOCALTONET_BOOTSTRAP_SERVERS = "rx93buflno.localto.net:3265"
LOCAL_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_BOOTSTRAP_SERVERS = LOCALTONET_BOOTSTRAP_SERVERS if USE_LOCALTONET else LOCAL_BOOTSTRAP_SERVERS
if USE_LOCALTONET and not LOCALTONET_BOOTSTRAP_SERVERS.strip():
    print("WARNING: Please set LOCALTONET_BOOTSTRAP_SERVERS before running stream cells.")

KAFKA_TOPIC = "iot-sensor-data"
STARTING_OFFSETS = "earliest"

# Delta database/tables
DB_NAME = "iot_analytics"
TARGET_TABLE = f"{DB_NAME}.smart_filtered_measurements"
STATE_TABLE = f"{DB_NAME}.smart_filter_state"

# Checkpoint location (DBFS or UC volume)
CHECKPOINT_PATH = "/Volumes/workspace/metrics_app_streaming/checkpoints/iot_smart_filtering"

# If True, remove old checkpoint to re-read from STARTING_OFFSETS
RESET_CHECKPOINT = False

# If True, clear state table so next run stores as first_record
RESET_STATE_TABLE = False

# Rule: max waiting time before forcing a record
MAX_INTERVAL_MINUTES = 15

# Databricks cluster nay khong ho tro infinite ProcessingTime trigger.
# Chon 1 trong 2 gia tri: 'availableNow' hoac 'once'
STREAM_TRIGGER = "availableNow"

# Print debug stats for each micro-batch
PRINT_BATCH_STATS = True

# Fallback threshold when metric_type has no specific config
DEFAULT_THRESHOLD = 0.5

## LocaltoNet Setup Notes
Use LocaltoNet TCP tunnel for Kafka and keep bootstrap endpoint consistent between Kafka advertised listener and Databricks notebook.

1. Start a LocaltoNet TCP tunnel to local Kafka port 9092.
2. Use this LocaltoNet endpoint: rx93buflno.localto.net:3265.
3. Restart Kafka with external advertised host/port set to that endpoint.
4. Set `LOCALTONET_BOOTSTRAP_SERVERS` in Cell 2 to `rx93buflno.localto.net:3265`.
5. For first run after endpoint change, set `RESET_CHECKPOINT = True`.

PowerShell example on local machine:
```powershell
$env:KAFKA_EXTERNAL_HOST = "rx93buflno.localto.net"
$env:KAFKA_EXTERNAL_PORT = "3265"
docker-compose up -d kafka
```

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.conf.set("spark.sql.shuffle.partitions", "200")

threshold_config = [
    ("temperature", 0.30),
    ("humidity", 1.00),
    ("soil_moisture", 1.00),
    ("light_intensity", 150.0),
    ("pressure", 0.20),
]
threshold_df = spark.createDataFrame(threshold_config, ["metric_type", "threshold"])

payload_schema = T.StructType([
    T.StructField("timestamp", T.StringType()),
    T.StructField("sensor_id", T.StringType()),
    T.StructField("location", T.StringType()),
    T.StructField("metric_type", T.StringType()),
    T.StructField("unit", T.StringType()),
    T.StructField("temperature", T.DoubleType()),
    T.StructField("humidity", T.DoubleType()),
    T.StructField("soil_moisture", T.DoubleType()),
    T.StructField("light_intensity", T.DoubleType()),
    T.StructField("pressure", T.DoubleType()),
])

In [ ]:
# ===== Read Kafka stream and normalize payload =====
kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", STARTING_OFFSETS)
    .option("failOnDataLoss", "false")
    .load()
)

parsed_df = (
    kafka_stream_df
    .select(
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.from_json(F.col("value").cast("string"), payload_schema).alias("payload")
    )
    .select("kafka_topic", "kafka_partition", "kafka_offset", "kafka_timestamp", "payload.*")
    .withColumn("event_ts", F.coalesce(F.to_timestamp("timestamp"), F.col("kafka_timestamp")))
    .withColumn(
        "metric_value",
        F.coalesce(
            F.col("temperature"),
            F.col("humidity"),
            F.col("soil_moisture"),
            F.col("light_intensity"),
            F.col("pressure")
        )
    )
    .filter(F.col("sensor_id").isNotNull())
    .filter(F.col("metric_type").isNotNull())
    .filter(F.col("metric_value").isNotNull())
    .select(
        "event_ts",
        "sensor_id",
        "location",
        "metric_type",
        "unit",
        F.col("metric_value").cast("double").alias("metric_value"),
        "kafka_topic",
        "kafka_partition",
        "kafka_offset"
    )
)

In [ ]:
# ===== Create Delta database and tables if needed =====
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  event_ts TIMESTAMP,
  sensor_id STRING,
  location STRING,
  metric_type STRING,
  metric_value DOUBLE,
  unit STRING,
  threshold DOUBLE,
  store_reason STRING,
  kafka_topic STRING,
  kafka_partition INT,
  kafka_offset LONG,
  processed_at TIMESTAMP
) USING DELTA
PARTITIONED BY (metric_type)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
  sensor_id STRING,
  metric_type STRING,
  last_saved_value DOUBLE,
  last_saved_at TIMESTAMP,
  unit STRING,
  location STRING,
  updated_at TIMESTAMP
) USING DELTA
""")

In [ ]:
# ===== Smart filtering logic in foreachBatch =====
def smart_filter_and_store(micro_batch_df, batch_id):
    batch_count = micro_batch_df.count()
    if PRINT_BATCH_STATS:
        print(f"[Batch {batch_id}] input rows = {batch_count}")
    if batch_count == 0:
        return

    # Giam tai: chi lay ban ghi moi nhat moi sensor+metric trong 1 micro-batch
    latest_window = Window.partitionBy("sensor_id", "metric_type").orderBy(
        F.col("event_ts").desc(),
        F.col("kafka_offset").desc()
)
    latest_df = (
        micro_batch_df
        .withColumn("rn", F.row_number().over(latest_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    state_df = spark.table(STATE_TABLE).select(
        "sensor_id",
        "metric_type",
        "last_saved_value",
        "last_saved_at"
    )

    evaluated_df = (
        latest_df.alias("m")
        .join(threshold_df.alias("t"), on="metric_type", how="left")
        .join(state_df.alias("s"), on=["sensor_id", "metric_type"], how="left")
        .withColumn("threshold", F.coalesce(F.col("t.threshold"), F.lit(DEFAULT_THRESHOLD)))
        .withColumn("value_diff", F.abs(F.col("m.metric_value") - F.col("s.last_saved_value")))
        .withColumn(
            "minutes_since_last",
            F.when(F.col("s.last_saved_at").isNull(), F.lit(None).cast("double"))
             .otherwise((F.col("m.event_ts").cast("long") - F.col("s.last_saved_at").cast("long")) / 60.0)
)
        .withColumn(
            "store_reason",
            F.when(F.col("s.last_saved_at").isNull(), F.lit("first_record"))
             .when(F.col("value_diff") >= F.col("threshold"), F.lit("threshold"))
             .when(F.col("minutes_since_last") >= F.lit(MAX_INTERVAL_MINUTES), F.lit("max_interval"))
             .otherwise(F.lit("skip"))
)
    )

    to_store_df = (
        evaluated_df
        .filter(F.col("store_reason") != "skip")
        .select(
            F.col("event_ts").alias("event_ts"),
            F.col("sensor_id").alias("sensor_id"),
            F.col("location").alias("location"),
            F.col("metric_type").alias("metric_type"),
            F.col("metric_value").alias("metric_value"),
            F.col("unit").alias("unit"),
            F.col("threshold").alias("threshold"),
            F.col("store_reason").alias("store_reason"),
            F.col("kafka_topic").alias("kafka_topic"),
            F.col("kafka_partition").alias("kafka_partition"),
            F.col("kafka_offset").alias("kafka_offset"),
            F.current_timestamp().alias("processed_at")
        )
    )

    to_store_count = to_store_df.count()
    if PRINT_BATCH_STATS:
        print(f"[Batch {batch_id}] rows after smart filter = {to_store_count}")
    if to_store_count == 0:
        return

    # 1) Append records that pass smart filter
    to_store_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)

    # 2) Upsert latest saved state
    state_updates_df = to_store_df.select(
        "sensor_id",
        "metric_type",
        F.col("metric_value").alias("last_saved_value"),
        F.col("event_ts").alias("last_saved_at"),
        "unit",
        "location",
        F.current_timestamp().alias("updated_at")
    )

    state_delta = DeltaTable.forName(spark, STATE_TABLE)
    (
        state_delta.alias("s")
        .merge(
            state_updates_df.alias("u"),
            "s.sensor_id = u.sensor_id AND s.metric_type = u.metric_type"
        )
        .whenMatchedUpdate(set={
            "last_saved_value": "u.last_saved_value",
            "last_saved_at": "u.last_saved_at",
            "unit": "u.unit",
            "location": "u.location",
            "updated_at": "u.updated_at"
        })
        .whenNotMatchedInsert(values={
            "sensor_id": "u.sensor_id",
            "metric_type": "u.metric_type",
            "last_saved_value": "u.last_saved_value",
            "last_saved_at": "u.last_saved_at",
            "unit": "u.unit",
            "location": "u.location",
            "updated_at": "u.updated_at"
        })
        .execute()
    )

    if PRINT_BATCH_STATS:
        print(f"[Batch {batch_id}] write completed.")

In [ ]:
# ===== Start streaming query =====
if RESET_CHECKPOINT:
    print(f"Removing checkpoint path: {CHECKPOINT_PATH}")
    dbutils.fs.rm(CHECKPOINT_PATH, True)

if RESET_STATE_TABLE:
    print(f"Truncating state table: {STATE_TABLE}")
    spark.sql(f"TRUNCATE TABLE {STATE_TABLE}")

# Stop old active query with the same name (if any)
for active_query in spark.streams.active:
    if active_query.name == "iot_smart_filtering_stream":
        print(f"Stopping existing query: {active_query.id}")
        active_query.stop()

writer = (
    parsed_df.writeStream
    .queryName("iot_smart_filtering_stream")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .foreachBatch(smart_filter_and_store)
)

trigger_mode = STREAM_TRIGGER.strip().lower()
if trigger_mode == "availablenow":
    writer = writer.trigger(availableNow=True)
elif trigger_mode == "once":
    writer = writer.trigger(once=True)
else:
    raise ValueError("STREAM_TRIGGER must be 'availableNow' or 'once' for this cluster type.")

query = writer.start()
query.awaitTermination()

print("Final query status:")
print(query.status)
if query.lastProgress is not None:
    print("Last progress:")
    print(query.lastProgress)

In [ ]:
# Monitoring / validation
display(
    spark.sql(
        f"""
        SELECT metric_type, COUNT(*) AS total_rows, MAX(event_ts) AS latest_event_ts
        FROM {TARGET_TABLE}
        GROUP BY metric_type
        ORDER BY metric_type
        """
    )
)

display(spark.sql(f"SELECT * FROM {TARGET_TABLE} ORDER BY event_ts DESC LIMIT 100"))
display(spark.sql(f"SELECT * FROM {STATE_TABLE} ORDER BY updated_at DESC LIMIT 100"))

## Stop stream

Chay cell sau khi muon dung stream:

```python
query.stop()
```